In [ ]:
import json

with open('output/output.json', 'r') as f:
    result = json.load(f)

In [ ]:
import matplotlib.pyplot as plt

# 设置中文字体，避免图表中的中文乱码
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'WenQuanYi Micro Hei']
plt.rcParams['axes.unicode_minus'] = False   # 解决负号显示问题

def plot_pie(data, target_name, title, pdf_path, labels=None, **kwargs):
    # 筛选匹配的数据项
    matched = [d for d in data if d.get('name') == target_name]
    if not matched:
        raise ValueError(f"未找到 name 为 '{target_name}' 的数据项")
    if len(matched) > 1:
        print(f"警告: 找到多个 name 为 '{target_name}' 的数据项，将使用第一个。")
    item = matched[0]
    
    # 提取 value 字典，过滤掉值为 0 的类别
    value_dict = item.get('value', {})
    if not value_dict:
        raise ValueError(f"数据项 '{target_name}' 的 value 字段为空")
    
    # 过滤值为 0 的类别（避免饼图中出现占空为 0 的扇区）
    filtered = {k: v for k, v in value_dict.items() if v != 0}
    if not filtered:
        raise ValueError(f"数据项 '{target_name}' 的所有 value 均为 0，无法绘制饼图")
    
    if labels is None:
        labels = list(filtered.keys())
    sizes = list(filtered.values())
    
    # 可选参数设置
    autopct = kwargs.get('autopct', '%1.1f%%')
    startangle = kwargs.get('startangle', 90)
    colors = kwargs.get('colors', None)
    show_legend = kwargs.get('show_legend', True)
    legend_loc = kwargs.get('legend_loc', 'best')
    label_distance = kwargs.get('label_distance', 1.1)
    
    # 创建图形和轴对象
    fig, ax = plt.subplots(figsize=(8, 6), dpi=150)
    
    # 绘制饼图
    wedges, texts, autotexts = ax.pie(
        sizes,
        labels=labels if not show_legend else None,  # 若显示图例，则不直接标在扇区上
        autopct=autopct,
        startangle=startangle,
        colors=colors,
        pctdistance=0.85,          # 百分比文字距离圆心的距离
        labeldistance=label_distance,
        textprops={'fontsize': 10}
    )
    
    # 美化百分比文字
    for autotext in autotexts:
        autotext.set_color('white')
        autotext.set_fontsize(9)
        autotext.set_weight('bold')
    
    # 添加图例
    if show_legend:
        # 创建包含 "类别: 占比% " 的图例标签
        total = sum(sizes)
        legend_labels = [f"{l} ({s/total*100:.3f}%)" for l, s in zip(labels, sizes)]
        ax.legend(wedges, legend_labels, title=title, loc=legend_loc, fontsize=9)
    
    # 设置标题
    ax.set_title(title, fontsize=14, pad=20)
    # 确保饼图为正圆
    ax.axis('equal')
    
    # 调整布局并保存为 PDF
    plt.tight_layout()
    plt.savefig(pdf_path, dpi=300, bbox_inches='tight', format='pdf')
    plt.close(fig)
    print(f"饼图已保存至 {pdf_path}")


with open('output/output.json', 'r') as f:
    result = json.load(f)

plot_pie(
    data=result,
    target_name='stats_richi_player_num',
    title='单局立直玩家数分布',
    pdf_path='output/richi_player_num.pdf',
    show_legend=True,
    label_distance=1.15
)

plot_pie(
    data=result,
    target_name='stats_round_end_type',
    title='单局的终局条件',
    pdf_path='output/round_end_type.pdf',
    show_legend=True,
    labels=['流局', '立直荣和', '立直自摸', '门清默听荣和', '门清默听自摸', '副露荣和', '副露自摸'],
    label_distance=1.15
)

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'WenQuanYi Micro Hei']
plt.rcParams['axes.unicode_minus'] = False   # 解决负号显示问题

def plot_num_occur_time(data, pdf_path='num_occur_time.pdf', name2title=None,
                        x_label='巡目数', y_label='发生次数', title='n巡目事件发生次数',
                        is_normalize=False):
    # 筛选 type == NUM_OCCUR_TIME
    occur_data = [d for d in data if d['type'] == 'NUM_OCCUR_TIME']
    if name2title is not None:
        occur_data = [d for d in occur_data if d['name'] in name2title]
        present = {d['name'] for d in occur_data}
        missing = set(name2title.keys()) - present
        assert not missing, f"数据中缺少以下名称: {missing}"
    assert len(occur_data) > 0, "没有符合条件的数据"

    # 收集所有键并排序
    all_keys = sorted({int(k) for item in occur_data for k in item['value']})
    x_labels = [str(k) for k in all_keys]

    stats_names, stats_values = [], []
    for item in occur_data:
        legend_name = name2title[item['name']] if name2title else item['name']
        stats_names.append(legend_name)
        stats_values.append([item['value'].get(str(k), 0) for k in all_keys])

    n_groups = len(all_keys)
    n_stats = len(stats_names)
    bar_width = 0.8 / n_stats
    index = np.arange(n_groups)
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

    fig, ax = plt.subplots(figsize=(14, 6), dpi=150)
    for i, (vals, name) in enumerate(zip(stats_values, stats_names)):
        weighted_data = [(k, v) for k, v in zip(all_keys, vals) if v > 0]
        total = sum(v for _, v in weighted_data)
        mean_val = sum(k * w for k, w in weighted_data) / total if total >= 1 else 0.0
        std_val = np.sqrt(sum(w * (k - mean_val) ** 2 for k, w in weighted_data) / (total - 1)) if total > 1 else 0.0
        legend_label = f"{name} ({mean_val:.2f}±{std_val:.2f})"

        normalized_vals = vals
        if is_normalize:
            vals_sum = sum(vals)
            normalized_vals = [val / vals_sum for val in vals]

        bars = ax.bar(index + i * bar_width, normalized_vals, bar_width,
                      label=legend_label, color=colors[i % len(colors)])

        for bar, normalized_val, val in zip(bars, normalized_vals, vals):
            if val > 0:
                val_str = '%.3f%% (%d)' % (normalized_val * 100, val) if is_normalize else '%.3f%%' % (normalized_val * 100)
                ax.text(bar.get_x() + bar.get_width() / 2.,
                        bar.get_height() + max(normalized_vals)*0.01,
                        val_str, ha='center', va='bottom',
                        fontsize=6, rotation=90)

    ax.set_xlabel(x_label)
    ax.set_ylabel(y_label)
    ax.set_title(title)
    ax.set_xticks(index + bar_width * (n_stats - 1) / 2)
    ax.set_xticklabels(x_labels)
    ax.legend(fontsize=8)
    ax.grid(axis='y', linestyle='--', alpha=0.7)

    plt.tight_layout()
    plt.savefig(pdf_path, dpi=300, bbox_inches='tight', format='pdf')
    plt.close(fig)
    print(f"图表已保存至 {pdf_path}")


with open('output/output.json', 'r') as f:
    result = json.load(f)


name_map = {
    'stats_game_round': '每个半庄麻将有n小局',
}
plot_num_occur_time(result, 'output/game_round.pdf', name2title=name_map, x_label='局数', y_label='发生比例', title='每个半庄麻将有n小局', is_normalize=True)


name_map = {
    'stats_richi_ok_num': 'n巡目立直成功玩家数',
    'stats_first_richi_ok_num': 'n巡目先制立直成功玩家数',
    'stats_chasing_richi_ok_num': 'n巡目追立直成功玩家数',
    'stats_be_chased_richi_ok_num': 'n巡目立直但被追立直成功玩家数',
}
plot_num_occur_time(result, 'output/richi_num.pdf', name2title=name_map, x_label='巡目数', y_label='发生次数', title='n巡目事件发生次数', is_normalize=True)


name_map = {
    'stats_agari_richi_ok_dora_num': 'n巡目立直成功宝牌数',
    'stats_agari_first_richi_ok_dora_num': 'n巡目先制立直成功宝牌数',
    'stats_agari_chasing_richi_ok_dora_num': 'n巡目追立直成功宝牌数',
    'stats_agari_be_chased_richi_ok_dora_num': 'n巡目立直但被追立直成功宝牌数',
}
plot_num_occur_time(result, 'output/richi_dora_num.pdf', name2title=name_map, x_label='表宝牌+红宝牌数量', y_label='发生次数', title='n巡目宝牌数-发生次数', is_normalize=True)

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'WenQuanYi Micro Hei']
plt.rcParams['axes.unicode_minus'] = False   # 解决负号显示问题

def plot_num_to_mean_std_sample(data, pdf_path='mean_std_sample.pdf', name2title=None,
                                x_label='巡目数', y_label='发生次数', title='n巡目事件发生次数',
                                offset=0.3):
    """
    将 NUM_TO_MEAN_STD_SAMPLE 数据绘制为带误差棒的折线图并保存为 PDF。
    参数 name2title: 字典，键为原始 name，值为图例显示名称。仅绘制 name 在字典中的项。
    参数 offset: 同一列上不同折线的水平偏移总量，默认0.15。设为0则不偏移。
    """
    # 筛选类型
    mean_std_data = [d for d in data if d['type'] == 'NUM_TO_MEAN_STD_SAMPLE']
    if name2title is not None:
        mean_std_data = [d for d in mean_std_data if d['name'] in name2title]
        present = {d['name'] for d in mean_std_data}
        missing = set(name2title.keys()) - present
        assert not missing, f"数据中缺少以下名称: {missing}"
    assert len(mean_std_data) > 0, "没有符合条件的数据"

    # 提取所有巡目并排序
    all_keys_std = sorted({int(k) for item in mean_std_data for k in item['value']})
    x_base = np.array(all_keys_std)          # 原始整数位置
    n_lines = len(mean_std_data)

    # 计算每条线的水平偏移量，使它们均匀分布在基准点左右
    if n_lines > 1 and offset > 0:
        shifts = np.linspace(-offset/2, offset/2, n_lines)
    else:
        shifts = [0.0] * n_lines

    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#17becf']
    markers = ['o', 's', '^', 'D', 'v', '<', '>', 'p', '*']

    fig, ax = plt.subplots(figsize=(14, 6), dpi=150)
    for idx, item in enumerate(mean_std_data):
        legend_name = name2title[item['name']] if name2title else item['name']

        # 计算期望begin: deepseek写的，我没改没review
        total_sum = 0.0
        weighted_sum = 0.0
        for k in all_keys_std:
            v = item['value'].get(str(k), None)
            if v is not None and v.get('total', 0) > 0:
                total_sum += v['total']
                weighted_sum += v['mean'] * v['total']
        if total_sum > 0:
            expected = weighted_sum / total_sum
            legend_name = f"{legend_name} ({expected:.2f})"
        # 计算期望end
        
        x_pos = x_base + shifts[idx]          # 偏移后的横坐标
        means, yerrs = [], []
        for k in all_keys_std:
            v = item['value'].get(str(k), None)
            if v is not None:
                means.append(v['mean'])
                sem = v['std'] / np.sqrt(v['total']) if v['total'] > 0 else 0
                yerrs.append(sem)
            else:
                means.append(np.nan)
                yerrs.append(0)

        ax.errorbar(x_pos, means, yerr=yerrs,
                    fmt=markers[idx % len(markers)] + '-',
                    color=colors[idx % len(colors)],
                    label=legend_name,
                    capsize=3, linewidth=1.5, markersize=5)

    ax.set_xlabel(x_label)
    ax.set_ylabel(y_label)
    ax.set_title(title)
    ax.set_xticks(x_base)                     # 刻度仍在原始整数位置
    ax.set_xticklabels(all_keys_std)
    ax.legend(fontsize=8)
    ax.grid(axis='y', linestyle='--', alpha=0.7)

    plt.tight_layout()
    plt.savefig(pdf_path, dpi=300, bbox_inches='tight', format='pdf')
    plt.close(fig)
    print(f"图表已保存至 {pdf_path}")


with open('output/output.json', 'r') as f:
    result = json.load(f)


name_map = {
    'stats_richi_n_ron_rate': 'n巡立直荣和率',
    'stats_richi_n_tsumo_rate': 'n巡立直自摸率',
    'stats_richi_n_be_ron_rate': 'n巡立直被荣和率',
    'stats_richi_n_be_tsumo_rate': 'n巡立直被自摸率',
    'stats_richi_n_draw_rate': 'n巡立直横移动率',
    'stats_richi_n_ryuukyoku_rate': 'n巡立直流局率',
}
plot_num_to_mean_std_sample(result, 'output/richi_result_rate.pdf', name2title=name_map, x_label='巡目数', y_label='发生次数', title='n巡目事件发生概率（带误差棒）')


name_map = {
    'stats_oya_richi_n_ron_rate': 'n巡庄家立直荣和率',
    'stats_oya_richi_n_tsumo_rate': 'n巡庄家立直自摸率',
    'stats_oya_richi_n_be_ron_rate': 'n巡庄家立直被荣和率',
    'stats_oya_richi_n_be_tsumo_rate': 'n巡庄家立直被自摸率',
    'stats_oya_richi_n_draw_rate': 'n巡庄家立直横移动率',
    'stats_oya_richi_n_ryuukyoku_rate': 'n巡庄家立直流局率',
}
plot_num_to_mean_std_sample(result, 'output/oya_richi_result_rate.pdf', name2title=name_map, x_label='巡目数', y_label='发生次数', title='n巡目事件发生概率（带误差棒）')


name_map = {
    'stats_not_oya_richi_n_ron_rate': 'n巡闲家立直荣和率',
    'stats_not_oya_richi_n_tsumo_rate': 'n巡闲家立直自摸率',
    'stats_not_oya_richi_n_be_ron_rate': 'n巡闲家立直被荣和率',
    'stats_not_oya_richi_n_be_tsumo_rate': 'n巡闲家立直被自摸率',
    'stats_not_oya_richi_n_draw_rate': 'n巡闲家立直横移动率',
    'stats_not_oya_richi_n_ryuukyoku_rate': 'n巡闲家立直流局率',
}
plot_num_to_mean_std_sample(result, 'output/not_oya_richi_result_rate.pdf', name2title=name_map, x_label='巡目数', y_label='发生次数', title='n巡目事件发生概率（带误差棒）')


name_map = {
    'stats_richi_n_gain': 'n巡立直局收支',
    'stats_richi_n_ron_gain': 'n巡立直荣和局收支',
    'stats_richi_n_tsumo_gain': 'n巡立直自摸局收支',
    'stats_richi_n_be_ron_gain': 'n巡立直被荣和局收支',
    'stats_richi_n_be_tsumo_gain': 'n巡立直被自摸局收支',
    'stats_richi_n_draw_gain': 'n巡立直横移动局收支',
    'stats_richi_n_ryuukyoku_gain': 'n巡立直流局局收支',
}
plot_num_to_mean_std_sample(result, 'output/richi_result_gain.pdf', name2title=name_map, x_label='巡目数', y_label='点棒差', title='n巡目事件发生时点棒差（带误差棒）')


name_map = {
    'stats_oya_richi_n_gain': 'n巡庄家立直局收支',
    'stats_oya_richi_n_ron_gain': 'n巡庄家立直荣和局收支',
    'stats_oya_richi_n_tsumo_gain': 'n巡庄家立直自摸局收支',
    'stats_oya_richi_n_be_ron_gain': 'n巡庄家立直被荣和局收支',
    'stats_oya_richi_n_be_tsumo_gain': 'n巡庄家立直被自摸局收支',
    'stats_oya_richi_n_draw_gain': 'n巡庄家立直横移动局收支',
    'stats_oya_richi_n_ryuukyoku_gain': 'n巡庄家立直流局局收支',
}
plot_num_to_mean_std_sample(result, 'output/oya_richi_result_gain.pdf', name2title=name_map, x_label='巡目数', y_label='点棒差', title='n巡目事件发生时点棒差（带误差棒）')


name_map = {
    'stats_not_oya_richi_n_gain': 'n巡闲家立直局收支',
    'stats_not_oya_richi_n_ron_gain': 'n巡闲家立直荣和局收支',
    'stats_not_oya_richi_n_tsumo_gain': 'n巡闲家立直自摸局收支',
    'stats_not_oya_richi_n_be_ron_gain': 'n巡闲家立直被荣和局收支',
    'stats_not_oya_richi_n_be_tsumo_gain': 'n巡闲家立直被自摸局收支',
    'stats_not_oya_richi_n_draw_gain': 'n巡闲家立直横移动局收支',
    'stats_not_oya_richi_n_ryuukyoku_gain': 'n巡闲家立直流局局收支',
}
plot_num_to_mean_std_sample(result, 'output/not_oya_richi_result_gain.pdf', name2title=name_map, x_label='巡目数', y_label='点棒差', title='n巡目事件发生时点棒差（带误差棒）')


name_map = {
    'stats_first_richi_n_ron_rate': 'n巡先制立直荣和率',
    'stats_first_richi_n_tsumo_rate': 'n巡先制立直自摸率',
    'stats_first_richi_n_be_ron_rate': 'n巡先制立直被荣和率',
    'stats_first_richi_n_be_tsumo_rate': 'n巡先制立直被自摸率',
    'stats_first_richi_n_draw_rate': 'n巡先制立直横移动率',
    'stats_first_richi_n_ryuukyoku_rate': 'n巡先制立直流局率',
}
plot_num_to_mean_std_sample(result, 'output/first_richi_result_rate.pdf', name2title=name_map, x_label='巡目数', y_label='发生次数', title='n巡目事件发生概率（带误差棒）')


name_map = {
    'stats_first_richi_n_gain': 'n巡先制立直局收支',
    'stats_first_richi_n_ron_gain': 'n巡先制立直荣和局收支',
    'stats_first_richi_n_tsumo_gain': 'n巡先制立直自摸局收支',
    'stats_first_richi_n_be_ron_gain': 'n巡先制立直被荣和局收支',
    'stats_first_richi_n_be_tsumo_gain': 'n巡先制立直被自摸局收支',
    'stats_first_richi_n_draw_gain': 'n巡先制立直横移动局收支',
    'stats_first_richi_n_ryuukyoku_gain': 'n巡先制立直流局局收支',
}
plot_num_to_mean_std_sample(result, 'output/first_richi_result_gain.pdf', name2title=name_map, x_label='巡目数', y_label='点棒差', title='n巡目事件发生时点棒差（带误差棒）')


name_map = {
    'stats_chasing_richi_n_ron_rate': 'n巡追立直荣和率',
    'stats_chasing_richi_n_tsumo_rate': 'n巡追立直自摸率',
    'stats_chasing_richi_n_be_ron_rate': 'n巡追立直被荣和率',
    'stats_chasing_richi_n_be_tsumo_rate': 'n巡追立直被自摸率',
    'stats_chasing_richi_n_draw_rate': 'n巡追立直横移动率',
    'stats_chasing_richi_n_ryuukyoku_rate': 'n巡追立直流局率',
}
plot_num_to_mean_std_sample(result, 'output/chasing_richi_result_rate.pdf', name2title=name_map, x_label='巡目数', y_label='发生次数', title='n巡目事件发生概率（带误差棒）')


name_map = {
    'stats_chasing_richi_n_gain': 'n巡追立直局收支',
    'stats_chasing_richi_n_ron_gain': 'n巡追立直荣和局收支',
    'stats_chasing_richi_n_tsumo_gain': 'n巡追立直自摸局收支',
    'stats_chasing_richi_n_be_ron_gain': 'n巡追立直被荣和局收支',
    'stats_chasing_richi_n_be_tsumo_gain': 'n巡追立直被自摸局收支',
    'stats_chasing_richi_n_draw_gain': 'n巡追立直横移动局收支',
    'stats_chasing_richi_n_ryuukyoku_gain': 'n巡追立直流局局收支',
}
plot_num_to_mean_std_sample(result, 'output/chasing_richi_result_gain.pdf', name2title=name_map, x_label='巡目数', y_label='点棒差', title='n巡目事件发生时点棒差（带误差棒）')


name_map = {
    'stats_be_chased_richi_n_ron_rate': 'n巡被追立直荣和率',
    'stats_be_chased_richi_n_tsumo_rate': 'n巡被追立直自摸率',
    'stats_be_chased_richi_n_be_ron_rate': 'n巡被追立直被荣和率',
    'stats_be_chased_richi_n_be_tsumo_rate': 'n巡被追立直被自摸率',
    'stats_be_chased_richi_n_draw_rate': 'n巡被追立直横移动率',
    'stats_be_chased_richi_n_ryuukyoku_rate': 'n巡被追立直流局率',
}
plot_num_to_mean_std_sample(result, 'output/be_chased_richi_result_rate.pdf', name2title=name_map, x_label='巡目数', y_label='发生次数', title='n巡目事件发生概率（带误差棒）')


name_map = {
    'stats_be_chased_richi_n_gain': 'n巡被追立直局收支',
    'stats_be_chased_richi_n_ron_gain': 'n巡被追立直荣和局收支',
    'stats_be_chased_richi_n_tsumo_gain': 'n巡被追立直自摸局收支',
    'stats_be_chased_richi_n_be_ron_gain': 'n巡被追立直被荣和局收支',
    'stats_be_chased_richi_n_be_tsumo_gain': 'n巡被追立直被自摸局收支',
    'stats_be_chased_richi_n_draw_gain': 'n巡被追立直横移动局收支',
    'stats_be_chased_richi_n_ryuukyoku_gain': 'n巡被追立直流局局收支',
}
plot_num_to_mean_std_sample(result, 'output/be_chased_richi_result_gain.pdf', name2title=name_map, x_label='巡目数', y_label='点棒差', title='n巡目事件发生时点棒差（带误差棒）')